# CT-TGNN — real experiments

Runs the full sweep on the real dataset with GPU.

**Before running:** Settings → Accelerator → **GPU**, and Internet → **On**.

Every number produced here is computed from model output written to
`runs/*/scores_test.npz`. No value is hardcoded anywhere in this pipeline.


## 1. Get the code and dependencies

In [ ]:
!git clone -q https://github.com/rogerpanel/CT-TGNN-Model-development.git repo || echo "already cloned"
%cd repo
!pip install -q torchdiffeq pyyaml psutil scipy
import torch; print("CUDA:", torch.cuda.is_available(),
                    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Inspect the dataset schema

Run this first. It reports the real columns so the loader binds to them
correctly. If any role shows `NOT FOUND`, set it explicitly in the config
in step 3.

In [ ]:
DATA = "/kaggle/input/integrated-idps-security-3datasets"
!ls -la {DATA}
!python -m src.data.inspect --path {DATA}

## 3. Configure

Edit `columns` below only if step 2 failed to detect a role. Start with a
row cap to confirm the pipeline runs, then remove it for the real run.

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["data"]["path"] = DATA
cfg["data"]["max_rows"] = 200_000     # set to None for the full dataset
cfg["data"]["window_seconds"] = 60.0

# Override only if auto-detection failed:
# cfg["data"]["columns"]["src_ip"] = "Src IP"
# cfg["data"]["columns"]["label"]  = "Label"

cfg["model"]["solver"] = "dopri5"     # adaptive; use "rk4" if too slow
cfg["model"]["use_adjoint"] = True
cfg["train"]["epochs"] = 20

yaml.safe_dump(cfg, open("configs/kaggle.yaml", "w"))
print(yaml.safe_dump(cfg))

## 4. Smoke run — one model, one seed

Confirms data loads, graphs build, and training converges before
committing to the full sweep.

In [ ]:
!python -m src.train --config configs/kaggle.yaml --model ct_tgnn --seed 0 --epochs 2

## 5. Full sweep

Six models x five seeds. Five seeds is the minimum for the variance and
significance testing reviewers asked for.

In [ ]:
!CONFIG=configs/kaggle.yaml SEEDS="0 1 2 3 4" bash scripts/run_all.sh

## 6. Generate tables and figures

Everything below is recomputed from the score arrays at generation time.
The ROC curve and its AUC come from the same array in the same call, so
they cannot disagree.

In [ ]:
!python -m src.report --runs runs --out paper --dataset-name "Integrated IDPS"
!cat paper/table_main.tex
print()
!cat paper/provenance.txt

In [ ]:
from IPython.display import IFrame, display
display(IFrame("paper/fig_roc.pdf", width=700, height=560))
display(IFrame("paper/fig_pr.pdf", width=700, height=560))

## 7. Export for DyGLib

To run TGN / TGAT / DyRep / JODIE / GraphMixer in their reference harness
under identical splits.

In [ ]:
!python scripts/export_dyglib.py --config configs/kaggle.yaml --out dyglib_export

## 8. Package results

Download `ct_tgnn_results.zip` and send it back — the tables, figures and
score files are what the paper gets written from.

In [ ]:
!zip -qr ct_tgnn_results.zip runs paper dyglib_export -x "*/model.pt"
!ls -lh ct_tgnn_results.zip
import json, glob
for f in sorted(glob.glob("runs/*/metrics.json")):
    m = json.load(open(f))
    print(f"{m['model']:<16} seed={m['seed']}  AUROC={m.get('auroc'):.4f}  "
          f"AUPRC={m.get('auprc'):.4f}  F1={m.get('f1'):.4f}")